# Fashion-MNIST CNN Classification

Three-way ablation study comparing:
- **Baseline** — Dense NN, no convolution
- **Plain CNN** — CNN, no augmentation
- **Augmented CNN** — CNN + data augmentation

Goal: isolate the contribution of (i) the architecture and (ii) data augmentation, instead of just reporting one number.

## 1. Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", "Yes" if tf.config.list_physical_devices('GPU') else "No (CPU only)")

## 2. Load Dataset

Fashion-MNIST: 70K grayscale 28x28 images, 10 clothing classes. Built into Keras.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {x_train.shape[0]}  Test: {x_test.shape[0]}")
print(f"Image size: {x_train.shape[1]}x{x_train.shape[2]}")
print(f"Classes: {len(class_names)}")

## 3. Quick Look

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(x_train[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[y_train[i]])
plt.suptitle("Fashion-MNIST samples", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Preprocessing

- Normalize pixels to `[0, 1]`
- Reshape `(28, 28)` → `(28, 28, 1)` so Conv2D gets a channel dim

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

print(f"Train shape: {x_train.shape}")
print(f"Test shape:  {x_test.shape}")

---

## 5. Baseline — Dense NN

Reference point. No convolution, no augmentation. Just a flat MLP.

In [ ]:
baseline_model = models.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='baseline_dense_nn')

baseline_model.compile(optimizer='adam',
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
baseline_model.summary()

5 epochs is enough — we only need a comparison reference, not a tuned model.

In [ ]:
baseline_history = baseline_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

baseline_loss, baseline_acc = baseline_model.evaluate(x_test, y_test, verbose=0)
print(f"\n>>> BASELINE test accuracy: {baseline_acc*100:.2f}%")

---

## 6. Plain CNN (no augmentation)

Same architecture as the augmented CNN later, just without the augmentation layer at the input. This isolates the architecture's contribution from augmentation's contribution.

In [ ]:
plain_cnn = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    # No augmentation layer here

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='plain_cnn')

plain_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
plain_cnn.summary()

15 epochs. Converges fast without augmentation.

In [ ]:
plain_history = plain_cnn.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

plain_loss, plain_acc = plain_cnn.evaluate(x_test, y_test, verbose=0)
print(f"\n>>> PLAIN CNN test accuracy: {plain_acc*100:.2f}%")

### Plain CNN training curves

If train and val accuracy diverge → overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(plain_history.history['accuracy'], label='train', marker='o')
axes[0].plot(plain_history.history['val_accuracy'], label='val', marker='s')
axes[0].set_title('Plain CNN — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(plain_history.history['loss'], label='train', marker='o')
axes[1].plot(plain_history.history['val_loss'], label='val', marker='s')
axes[1].set_title('Plain CNN — Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Plain CNN (no augmentation) training', fontsize=12)
plt.tight_layout()
plt.show()

---

## 7. Data Augmentation

Random rotation / zoom / translation, applied at training time only (test time it's a no-op). Goal: regularize by feeding slightly different versions each epoch.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),       # ~ +/- 18 degrees
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.03, 0.03),
], name='data_augmentation')

# Visualize what augmentation actually does
sample = x_train[0:1]
plt.figure(figsize=(12, 3))
plt.subplot(1, 6, 1); plt.imshow(sample[0].squeeze(), cmap='binary')
plt.title("Original"); plt.xticks([]); plt.yticks([])

for i in range(5):
    augmented = data_augmentation(sample, training=True)
    plt.subplot(1, 6, i + 2)
    plt.imshow(augmented[0].numpy().squeeze(), cmap='binary')
    plt.title(f"Aug {i+1}"); plt.xticks([]); plt.yticks([])

plt.suptitle("Same image, different augmentations")
plt.tight_layout()
plt.show()

## 8. Augmented CNN

Same conv stack as Plain CNN — but augmentation goes first.

In [ ]:
aug_cnn = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='aug_cnn')

aug_cnn.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
aug_cnn.summary()

## 9. Train Augmented CNN

30 epochs — augmented model converges slower since each epoch sees different variants.

In [ ]:
aug_history = aug_cnn.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

## 10. Augmented CNN training curves

Compare against the Plain CNN curves above.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(aug_history.history['accuracy'], label='train', marker='o')
axes[0].plot(aug_history.history['val_accuracy'], label='val', marker='s')
axes[0].set_title('Augmented CNN — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(aug_history.history['loss'], label='train', marker='o')
axes[1].plot(aug_history.history['val_loss'], label='val', marker='s')
axes[1].set_title('Augmented CNN — Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Augmented CNN test accuracy

In [ ]:
aug_loss, aug_acc = aug_cnn.evaluate(x_test, y_test, verbose=0)
print(f"Augmented CNN test loss:     {aug_loss:.4f}")
print(f"Augmented CNN test accuracy: {aug_acc*100:.2f}%")

---

## 12. Three-Way Comparison

The whole point of the ablation:
- `Baseline → Plain CNN` = contribution of the **architecture**
- `Plain CNN → Augmented CNN` = contribution of **augmentation**

In [ ]:
results = {
    'Baseline\n(Dense NN)':       (baseline_acc, baseline_model.count_params()),
    'Plain CNN\n(no aug)':        (plain_acc,    plain_cnn.count_params()),
    'Augmented CNN\n(with aug)':  (aug_acc,      aug_cnn.count_params()),
}

# Table
print("=" * 70)
print(f"{'Model':<30}{'Test Accuracy':>18}{'Params':>14}")
print("=" * 70)
for name, (acc, p) in results.items():
    flat = name.replace('\n', ' ')
    print(f"{flat:<30}{acc*100:>15.2f}%   {p:>12,}")
print("=" * 70)

# Decompose
arch_gain = (plain_acc - baseline_acc) * 100
aug_gain  = (aug_acc - plain_acc) * 100
total     = (aug_acc - baseline_acc) * 100

print(f"\nCONTRIBUTION DECOMPOSITION")
print(f"-----------------------------------------------------")
print(f"Architecture (Baseline -> Plain CNN):  {arch_gain:+.2f} pts")
print(f"Augmentation (Plain CNN -> Aug CNN):   {aug_gain:+.2f} pts")
print(f"Total        (Baseline -> Aug CNN):    {total:+.2f} pts")

# Bar chart with arrows showing contributions
fig, ax = plt.subplots(figsize=(9, 6))
labels = list(results.keys())
accs = [v[0] * 100 for v in results.values()]
ax.bar(labels, accs, color=['#888888', '#5588bb', '#2a5285'], edgecolor='black')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Three-Way Model Comparison')
ax.set_ylim([min(accs) - 2, max(accs) + 2])
ax.grid(True, axis='y', alpha=0.3)

for i, a in enumerate(accs):
    ax.text(i, a + 0.15, f'{a:.2f}%', ha='center', fontsize=12, fontweight='bold')

ax.annotate('', xy=(1, accs[1]), xytext=(0, accs[0]),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
ax.text(0.5, (accs[0] + accs[1])/2 + 0.3,
        f'Architecture\n{arch_gain:+.2f}', ha='center',
        color='green', fontsize=10, fontweight='bold')

ax.annotate('', xy=(2, accs[2]), xytext=(1, accs[1]),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.text(1.5, (accs[1] + accs[2])/2 + 0.3,
        f'Augmentation\n{aug_gain:+.2f}', ha='center',
        color='blue', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 13. Sample Predictions

Blue = correct, Red = wrong. Using the augmented CNN for the rest of the analysis (error patterns are basically the same across both CNNs).

In [ ]:
# Use augmented CNN for the analysis below
predictions = aug_cnn.predict(x_test, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

plt.figure(figsize=(15, 9))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(x_test[i].reshape(28, 28), cmap=plt.cm.binary)
    pred = predicted_classes[i]
    true = y_test[i]
    conf = 100 * np.max(predictions[i])
    color = 'blue' if pred == true else 'red'
    plt.xlabel(f"Pred: {class_names[pred]} ({conf:.1f}%)\nTrue: {class_names[true]}",
               color=color, fontsize=10)
plt.suptitle("Sample predictions on test set", fontsize=14)
plt.tight_layout()
plt.show()

## 14. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nPer-class report:\n")
print(classification_report(y_test, predicted_classes, target_names=class_names))

---

## 15. Misclassification Analysis

Beyond accuracy: what does the model actually get wrong, and is the pattern meaningful?

In [ ]:
wrong_idx = np.where(predicted_classes != y_test)[0]
print(f"Total test samples: {len(y_test)}")
print(f"Misclassified: {len(wrong_idx)} ({len(wrong_idx)/len(y_test)*100:.2f}%)")

# Most confidently wrong predictions — these are the trickiest cases
wrong_conf = np.max(predictions[wrong_idx], axis=1)
most_confident_wrong = wrong_idx[np.argsort(wrong_conf)[-12:][::-1]]

plt.figure(figsize=(15, 9))
for i, idx in enumerate(most_confident_wrong):
    plt.subplot(3, 4, i + 1)
    plt.xticks([]); plt.yticks([])
    plt.imshow(x_test[idx].reshape(28, 28), cmap=plt.cm.binary)
    pred = predicted_classes[idx]
    true = y_test[idx]
    conf = 100 * predictions[idx][pred]
    plt.xlabel(f"Pred: {class_names[pred]} ({conf:.1f}%)\nTrue: {class_names[true]}",
               color='red', fontsize=9)

plt.suptitle("Most confidently WRONG predictions", fontsize=13)
plt.tight_layout()
plt.show()

### Most confused class pairs

In [ ]:
# Zero out diagonal to find off-diagonal hotspots
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

most_confused = []
for _ in range(5):
    idx = np.unravel_index(np.argmax(cm_off), cm_off.shape)
    most_confused.append((idx[0], idx[1], cm_off[idx]))
    cm_off[idx] = 0

print("Most confused class pairs:\n")
print(f"{'True':<20}{'-->':<8}{'Predicted':<20}{'Count':>8}")
print("-" * 60)
for true, pred, count in most_confused:
    print(f"{class_names[true]:<20}{'-->':<8}{class_names[pred]:<20}{count:>8}")

### Interpretation

The errors aren't random — they cluster around visually similar classes (Shirt vs T-shirt, Coat vs Pullover). At 28x28 grayscale these are genuinely hard, even for humans. This is a known characteristic of Fashion-MNIST, not a bug in the model.

Could be improved with: higher resolution images, RGB color, or transfer learning from a larger pretrained network.

---

## 16. Conclusion & Save

### Summary

| Stage | Result |
|-------|--------|
| Baseline (Dense NN) | ~86.6% |
| Plain CNN | **~91.0%** ⭐ best |
| Augmented CNN | ~89.1% |

### Takeaways
1. CNN beats Dense by ~4 points using **fewer** parameters → architecture matters more than scale.
2. Augmentation **didn't help** here. With 60K balanced training images and a model that wasn't overfitting (train ≈ val), augmentation just adds noise. Not a universal good.
3. Errors cluster around visually similar classes (Shirt ↔ T-shirt, Coat ↔ Pullover) → model learned meaningful features.

### Save the best model

In [ ]:
# Plain CNN was our best, save it
plain_cnn.save('fashion_mnist_cnn.keras')
print("Saved as 'fashion_mnist_cnn.keras'")